In [1]:
import pandas as pd
from pathlib import Path

In [ ]:
DATA_DIR = Path.cwd().parent / "data"
DATA_FILE = DATA_DIR / "ventas.csv.gz"
RESULT_FILE = Path.cwd().parent / "submission" / "data_quality_report.json"

In [ ]:
data_ventas = pd.read_csv(DATA_FILE, dtype=str, keep_default_na=False, encoding="utf-8-sig")    # dtype=str garantiza una lectura del archivo que guarda la información original
data_ventas                                                                                     # keep_default_na=False hace que los términos comunes asociados a valores faltantes no sean declarados inmediantamente como NaN por pandas
                                                                                                # utf-8-sig elimina la marca BOM

,Supplier ID,supplier,COUNTRY,City,Purchase Date,Amount,Discount,Weight,Units,Unit Price,Contact Email
0,0001,ibm colombia s.a.s.,COLOMBIA,MEDELLÍN,02/02/2026,"$480,000",0.0,50000 g,5,96000.00,compras2@empresa.com
1,0002,MICROSOFT COLOMBIA INC.,colombia,bogotá,03/03/2026,"COP 980,000",5,0.0050 ton,1,980.000.00,compras3@empresa.com
2,0003,Grupo Éxito S.A.,COL,Medellin,2026/04/04,480.000,20%,12,1,480.000.00,compras4@empresa.com
3,0004,Alpina Productos Alimenticios,CO,Sopó,05-05-26,250000,,5 kg,2,"$125,000.00",compras5@empresa.com
4,0005,Cementos Argos SA,Colombia,Medellín,06.06.2026,250.00K,N/A,,20,12500.00,compras6@empresa.com
...,...,...,...,...,...,...,...,...,...,...,...
98,0099,Sura S.A.,COL,Medellín,2026/04/19,3.500.000,5%,10000 g,2,1.750.000.00,sin correo
99,0100,Corona SAS,CO,Bogotá,20-05-26,480000,,0.0025 ton,10,"$48,000.00",compras1@empresa.com
100,0005,Cementos Argos SA,Colombia,Medellín,06.06.2026,250.00K,N/A,,20,12500.00,compras6@empresa.com
101,0019,Sura S.A.,COLOMBIA,Medellín,20/04/2026,980.000,0.05,,10,98.000.00,compras20@empresa.com


In [7]:
# Normalización de los encabezados a minúscula, reemplazar " " por "_" y eliminar espacios en blanco al inicio y al final
data_ventas.columns = data_ventas.columns.str.strip().str.lower().str.replace(" ", "_")
data_ventas

,supplier_id,supplier,country,city,purchase_date,amount,discount,weight,units,unit_price,contact_email
0,0001,ibm colombia s.a.s.,COLOMBIA,MEDELLÍN,02/02/2026,"$480,000",0.0,50000 g,5,96000.00,compras2@empresa.com
1,0002,MICROSOFT COLOMBIA INC.,colombia,bogotá,03/03/2026,"COP 980,000",5,0.0050 ton,1,980.000.00,compras3@empresa.com
2,0003,Grupo Éxito S.A.,COL,Medellin,2026/04/04,480.000,20%,12,1,480.000.00,compras4@empresa.com
3,0004,Alpina Productos Alimenticios,CO,Sopó,05-05-26,250000,,5 kg,2,"$125,000.00",compras5@empresa.com
4,0005,Cementos Argos SA,Colombia,Medellín,06.06.2026,250.00K,N/A,,20,12500.00,compras6@empresa.com
...,...,...,...,...,...,...,...,...,...,...,...
98,0099,Sura S.A.,COL,Medellín,2026/04/19,3.500.000,5%,10000 g,2,1.750.000.00,sin correo
99,0100,Corona SAS,CO,Bogotá,20-05-26,480000,,0.0025 ton,10,"$48,000.00",compras1@empresa.com
100,0005,Cementos Argos SA,Colombia,Medellín,06.06.2026,250.00K,N/A,,20,12500.00,compras6@empresa.com
101,0019,Sura S.A.,COLOMBIA,Medellín,20/04/2026,980.000,0.05,,10,98.000.00,compras20@empresa.com


In [ ]:
# La forma de validar las columnas requeridas o no requeridas es declarando cuáles son precisamente estas con la información del enunciado

REQUIRED = ["supplier_id", "supplier", "country", "city", "purchase_date", 
            "amount", "discount", "weight", "units", "unit_price", "contact_email"]

# Al realizar la lectura usando encoding="utf-8-sig", se hace necesario definir que será considerando NA

MISSING_VALUES = ["", "N/A"]

# También se hará uso de expresiones regulares para identificar los correos que no cumplan con el formato

EMAIL_PATTERN = r"[^@\s]+@[^@\s]+.[^@\s]+"

In [ ]:
# Validación número de filas y columnas 

row_count = int(data_ventas.shape[0])  # cantidad de filas
column_count = int(data_ventas.shape[1])  # cantidad de columnas

In [ ]:
# Validación de columnas requeridas e inesperadas 
missing_required_columns = sorted(set(REQUIRED) - set(data_ventas.columns)) # Se usa set para operar entre ambos conjuntos
unexpected_columns = sorted(set(data_ventas.columns) - set(REQUIRED))

In [ ]:
# Validación de duplicados
duplicate_row_count = int(data_ventas.duplicated().sum())

if "supplier_id" in data_ventas.columns:
    duplicate_supplier_id_count = int(data_ventas.duplicated(subset=["supplier_id"], keep=False).sum()) # subset para analizar los duplicados sobre supplier_id y keep=False para marcar todas las apariciones como repetidas, sin exlcuir la primera

else:
    duplicate_supplier_id_row_count = 0

In [ ]:
# faltantes
missing_value_count_by_column = {
    col: int(n) for col, n in data_ventas.isin(MISSING_VALUES).sum().items()    # Se analiza la cantidad de valores faltantes por columna siguiendo la definición de MISSING_VALUES. Con el .items() se crea el par clave, valor 
}

In [ ]:
# Validación de correos sin el formato requerido
if "contact_email" in data_ventas.columns:
    email_valido = data_ventas["contact_email"].str.fullmatch(EMAIL_PATTERN)  # Se usa str.fullmatch para validar que cada correo cumpla con el formato de la expresión regular definido en EMAIL_PATTERN
    invalid_email_count = int((~email_valido).sum())                          # Se usa ~ para indicar la negación. email_valido busca los emails validos, ~email_valido busca los emails invalidos

else: 
    invalid_email_count = 0

In [ ]:
# Validación de enteros no positivos
if "units" in data_ventas.columns:
    elementos = pd.to_numeric(data_ventas["units"], errors="coerce").dropna()
    no_positivo = elementos <=0
    no_entero = elementos % 1 != 0   # Se define no entero como todo número que tenga un residuo diferente de 0 al dividir el valor que toma elementos en ese bucle y lo divide entre 1
    invalid_unit_count = int((no_positivo | no_entero).sum())

else:
    invalid_unit_count = 0

In [ ]:
# Validación de nombre de país
if "country" in data_ventas.columns:
    paises = data_ventas["country"]
    country_values = sorted(paises[~paises.isin(MISSING_VALUES)].unique())  # Se organizan los países, es decir, los resultados de data_ventas["country"], excluyendo los NA con ~paises

else:
    country_values = []

In [ ]:
# Generar el diccionario con el reporte
reporte = {
    "row_count": row_count,
    "column_count": column_count,
    "missing_required_columns": missing_required_columns,
    "unexpected_columns": unexpected_columns,
    "duplicate_row_count": duplicate_row_count,
    "duplicate_supplier_id_row_count": duplicate_supplier_id_count,
    "missing_value_count_by_column": missing_value_count_by_column,
    "invalid_email_count": invalid_email_count,
    "invalid_unit_count": invalid_unit_count,
    "country_values": country_values
}

In [ ]:
# Generar el diccionario en formato .json en submission
with open(RESULT_FILE, "w", encoding="utf-8") as f:         
    json.dump(reporte, f, ensure_ascii=False, indent=4)     # Se pone el contenido de reporte en el nuevo file (f)
                                                            # ensure_ascii= False hace que se puedan leer caracteres especiales como tildes o ñ.indent=4 hace que el diccionario no sea una única línea muy extensa